In [ ]:
!pip install tensorboardX rdkit
!pip install rdkit-pypi

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as Data
import numpy as np
import time
import gc
import sys
import pickle
import copy
import pandas as pd

from sklearn.model_selection import train_test_split, LeaveOneOut, ParameterSampler
from sklearn.metrics import (roc_auc_score, matthews_corrcoef, recall_score, 
                           accuracy_score, r2_score, mean_squared_error, 
                           mean_absolute_error, precision_score, 
                           precision_recall_curve, auc, f1_score, roc_curve)

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib
from IPython.display import SVG, display
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import QED
from numpy.polynomial.polynomial import polyfit
from collections import defaultdict
from tensorboardX import SummaryWriter

from GCN import GCNModel, save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight

# Configure PyTorch and environment
torch.manual_seed(8)
sys.path.append('/notebooks/data/')
sys.setrecursionlimit(50000)
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
torch.nn.Module.dump_patches = True
sns.set(color_codes=True)
%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device(type='cuda')

In [ ]:
def prepare_model_and_data(raw_filename, target_name='Calx', targets=None, random_seed=888, 
                          batch_size=50, epochs=800, p_dropout=0.1, fingerprint_dim=150, 
                          weight_decay=2, learning_rate=3, radius=3, T=2, per_target_output_units_num=1):
    """
    Prepare GCN model and process molecular data for training.
    
    Args:
        raw_filename (str): Path to the CSV file containing SMILES and targets
        target_name (str): Name of the target column (default: 'Calx')
        targets (list): List of target column names
        random_seed (int): Random seed for reproducibility
        batch_size (int): Batch size for training
        epochs (int): Number of training epochs
        p_dropout (float): Dropout probability
        fingerprint_dim (int): Dimension of molecular fingerprints
        weight_decay (int): Weight decay parameter (as power of 10)
        learning_rate (int): Learning rate parameter (as power of 10)
        radius (int): Radius for molecular graph convolution
        T (int): Number of message passing iterations
        per_target_output_units_num (int): Output units per target
        
    Returns:
        tuple: (model, optimizer, loss_function, processed_dataframe, targets, feature_dicts)
    """
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    feature_filename = raw_filename.replace('.csv', '.pickle')
    filename = raw_filename.replace('.csv', '')
    prefix_filename = raw_filename.split('/')[-1].replace('.csv', '')
    smiles_targets_df = pd.read_csv(raw_filename)

    smilesList = smiles_targets_df.SMILES.values
    print("number of all smiles: ", len(smilesList))

    atom_num_dist = []
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            atom_num_dist.append(len(mol.GetAtoms()))
            remained_smiles.append(smiles)
            canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
        except:
            print(smiles)
            pass
    print("number of successfully processed smiles: ", len(remained_smiles))

    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    start_time = str(time.ctime()).replace(':', '-').replace(' ', '_')
    output_units_num = len(targets) * per_target_output_units_num

    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = save_smiles_dicts(smilesList, filename)
    feature_dicts = get_smiles_dicts(smilesList)

    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, smiles_to_rdkit_list = get_smiles_array([canonical_smiles_list[0]], feature_dicts)
    num_atom_features = x_atom.shape[-1]
    num_bond_features = x_bonds.shape[-1]

    loss_function = nn.MSELoss()
    model = GCNModel(radius, T, num_atom_features, num_bond_features, fingerprint_dim, output_units_num, p_dropout)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)

    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])

    return model, optimizer, loss_function, remained_df, targets, feature_dicts

In [ ]:
def train(model, dataset, optimizer, loss_function, epoch, batch_size, targets, feature_dicts, ratio_list):
    """
    Train the GCN model for one epoch.
    
    Args:
        model: GCN model to train
        dataset: Training dataset
        optimizer: PyTorch optimizer
        loss_function: Loss function to use
        epoch: Current epoch number
        batch_size: Size of training batches
        targets: List of target column names
        feature_dicts: Dictionary containing molecular features
        ratio_list: List of ratios for weighting losses
        
    Returns:
        float: Average training loss for the epoch
    """
    model.train()
    np.random.seed(epoch)
    
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]
    else:
        valList = np.arange(0, dataset.shape[0])
        np.random.shuffle(valList)
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    total_loss = 0
    
    for counter, batch in enumerate(batch_list):
        batch_df = dataset.loc[batch, :]
        smiles_list = batch_df.cano_smiles.values
        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(smiles_list, feature_dicts)
        
        x_atom = torch.Tensor(x_atom).to(device)
        x_bonds = torch.Tensor(x_bonds).to(device)
        x_atom_index = torch.LongTensor(x_atom_index).to(device)
        x_bond_index = torch.LongTensor(x_bond_index).to(device)
        x_mask = torch.Tensor(x_mask).to(device)
        
        atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
        
        optimizer.zero_grad()
        loss = 0.0
        
        for i, target in enumerate(targets):
            y_pred = mol_prediction[:, i]
            y_val = torch.Tensor(batch_df[target].values).squeeze().to(device)
            target_loss = loss_function(y_pred, y_val) * (ratio_list[i] ** 2)
            loss += target_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(batch_list)


def eval(model, dataset, targets, feature_dicts, batch_size):
    """
    Evaluate the GCN model on a dataset.
    
    Args:
        model: GCN model to evaluate
        dataset: Dataset to evaluate on
        targets: List of target column names
        feature_dicts: Dictionary containing molecular features
        batch_size: Size of evaluation batches
        
    Returns:
        tuple: (MAE_array, MSE_array, predictions_dict, actuals_dict, smiles_list)
    """
    model.eval()
    
    eval_MAE_list = {target: [] for target in targets}
    eval_MSE_list = {target: [] for target in targets}
    y_val_list = {target: [] for target in targets}
    y_pred_list = {target: [] for target in targets}
    smiles_list = []
    
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]
    else:
        valList = np.arange(0, dataset.shape[0])
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for batch in batch_list:
            batch_df = dataset.loc[batch, :]
            batch_smiles = batch_df.cano_smiles.values
            smiles_list.extend(batch_smiles)
            
            x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(batch_smiles, feature_dicts)
            
            x_atom = torch.Tensor(x_atom).to(device)
            x_bonds = torch.Tensor(x_bonds).to(device)
            x_atom_index = torch.LongTensor(x_atom_index).to(device)
            x_bond_index = torch.LongTensor(x_bond_index).to(device)
            x_mask = torch.Tensor(x_mask).to(device)
            
            atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
            
            for i, target in enumerate(targets):
                y_pred = mol_prediction[:, i].view(-1, 1)
                y_val = torch.Tensor(batch_df[target].values.reshape(-1, 1)).to(device)
                
                MAE = F.l1_loss(y_pred, y_val, reduction='none')
                MSE = F.mse_loss(y_pred, y_val, reduction='none')
                
                y_pred_list[target].extend(y_pred.cpu().numpy().flatten())
                y_val_list[target].extend(y_val.cpu().numpy().flatten())
                
                eval_MAE_list[target].extend(MAE.cpu().numpy().flatten())
                eval_MSE_list[target].extend(MSE.cpu().numpy().flatten())
    
    eval_MAE = np.array([np.mean(eval_MAE_list[target]) for target in targets])
    eval_MSE = np.array([np.mean(eval_MSE_list[target]) for target in targets])
    
    return eval_MAE, eval_MSE, y_pred_list, y_val_list, smiles_list



def train_and_evaluate(model, remained_df, targets, feature_dicts, optimizer, loss_function, 
                      batch_size, num_epochs, patience=30, min_delta=0.001, 
                      prefix_filename='', start_time=''):
    """
    Perform Leave-One-Out cross-validation training and evaluation.
    
    Args:
        model: GCN model to train
        remained_df: Processed dataset DataFrame
        targets: List of target column names
        feature_dicts: Dictionary containing molecular features
        optimizer: PyTorch optimizer
        loss_function: Loss function to use
        batch_size: Size of training/evaluation batches
        num_epochs: Maximum number of training epochs
        patience: Early stopping patience
        min_delta: Minimum improvement threshold for early stopping
        prefix_filename: Prefix for output files (unused)
        start_time: Start time string (unused)
        
    Returns:
        dict: Dictionary containing host predictions, overall metrics, fold results, and predictions
    """
    # Initialize tracking structures
    fold_results = []
    regression = {}
    best_param = {
        "train_epoch": 0, 
        "val_epoch": 0, 
        "train_MSE": float('inf'), 
        "val_MSE": float('inf')
    }
    
    # Prediction containers
    prediction_containers = {
        'train': {t: {'pred': [], 'actual': []} for t in targets},
        'test': {t: {'pred': [], 'actual': []} for t in targets}
    }
    
    # Cross-validation setup
    loo = LeaveOneOut()
    initial_state = model.state_dict()
    host_results = {}
    
    for fold, (train_index, test_index) in enumerate(loo.split(remained_df)):
        # --- Fold Initialization ---
        # Reset model and optimizer
        model.load_state_dict(initial_state)
        optimizer = type(optimizer)(model.parameters(), **optimizer.defaults)  # Proper reset
        
        # Data splits
        train_df = remained_df.iloc[train_index]
        test_df = remained_df.iloc[test_index]
        current_smile = test_df['Host'].values[0]
        
        # Validation split with minimum size
        val_size = max(5, int(0.1 * len(train_df)))
        train_df, val_df = train_test_split(
            train_df, 
            test_size=val_size,
            stratify=pd.qcut(train_df[targets[0]], q=5) if len(targets) > 0 else None
        )
        
        # --- Training Loop ---
        best_val_mse = float('inf')
        epochs_no_improve = 0
        best_model_state = None
        best_val_metrics = None
        
        for epoch in range(num_epochs):
            # Training phase
            train_loss = train(model, train_df, optimizer, loss_function, epoch, 
                              batch_size, targets, feature_dicts, ratio_list=[1.0]*len(targets))
            
            # Validation evaluation
            val_metrics = eval(model, val_df, targets, feature_dicts, batch_size)
            val_mae, val_mse = val_metrics[0].mean(), val_metrics[1].mean()
            
            # Model checkpointing based on validation
            if val_mse < best_val_mse - min_delta:
                best_val_mse = val_mse
                epochs_no_improve = 0
                best_model_state = model.state_dict()
                best_val_metrics = val_metrics
                best_param.update({
                    "val_epoch": epoch,
                    "val_MSE": val_mse,
                    "train_epoch": epoch,
                    "train_MSE": train_loss  # Using actual training loss
                })
            else:
                epochs_no_improve += 1
            
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
        
        # --- Post-Training Evaluation ---
        # Load best model
        model.load_state_dict(best_model_state)
        
        # Re-evaluate all sets with best model
        final_train_metrics = eval(model, train_df, targets, feature_dicts, batch_size)
        final_val_metrics = best_val_metrics
        test_metrics = eval(model, test_df, targets, feature_dicts, batch_size)
        test_MAE, test_MSE, test_pred, test_actual, _ = test_metrics
        
        # Store results by host
        host_results[current_smile] = {
            target: {
                "actual": float(test_actual[target][0]),  # Convert to Python float
                "predicted": float(test_pred[target][0])
            }
            for target in targets
        }
        # Store predictions
        for target in targets:
            # Training data (from final evaluation)
            prediction_containers['train'][target]['pred'].extend(final_train_metrics[2][target])
            prediction_containers['train'][target]['actual'].extend(final_train_metrics[3][target])
            
            # Test data
            prediction_containers['test'][target]['pred'].extend(test_metrics[2][target])
            prediction_containers['test'][target]['actual'].extend(test_metrics[3][target])
        
        # --- Fold Bookkeeping ---
        fold_results.append({
            'fold': fold + 1,
            'smiles': current_smile,
            'train_metrics': {
                'MAE': final_train_metrics[0].tolist(),
                'MSE': final_train_metrics[1].tolist()
            },
            'val_metrics': {
                'MAE': final_val_metrics[0].tolist(),
                'MSE': final_val_metrics[1].tolist()
            },
            'test_metrics': {
                'MAE': test_metrics[0].tolist(),
                'MSE': test_metrics[1].tolist()
            }
        })
        
        print(f"\nFold {fold+1} Summary:")
        print(f"Train MSE: {np.mean(final_train_metrics[1]):.4f}")
        print(f"Val MSE: {np.mean(final_val_metrics[1]):.4f}")
        print(f"Test MSE: {np.mean(test_metrics[1]):.4f}")
    
    # --- Final Aggregation ---
    def calculate_overall(metric_key):
        train_vals = [np.mean(f['train_metrics'][metric_key]) for f in fold_results]
        test_vals = [np.mean(f['test_metrics'][metric_key]) for f in fold_results]
        return np.mean(train_vals), np.mean(test_vals)
    
    overall_train_mae, overall_test_mae = calculate_overall('MAE')
    overall_train_mse, overall_test_mse = calculate_overall('MSE')
    
    print("\nFinal Performance:")
    print(f"Train MAE: {overall_train_mae:.4f} ± {np.std([f['train_metrics']['MAE'] for f in fold_results]):.4f}")
    print(f"Test MAE: {overall_test_mae:.4f} ± {np.std([f['test_metrics']['MAE'] for f in fold_results]):.4f}")
    print(f"Train MSE: {overall_train_mse:.4f} ± {np.std([f['train_metrics']['MSE'] for f in fold_results]):.4f}")
    print(f"Test MSE: {overall_test_mse:.4f} ± {np.std([f['test_metrics']['MSE'] for f in fold_results]):.4f}")
    
    return {
        "host_predictions": host_results,  # Your primary required output
        "overall_metrics": (overall_train_mae, overall_train_mse, 
                           overall_test_mae, overall_test_mse),
        "fold_results": fold_results,
        "train_predictions": prediction_containers['train'],
        "test_predictions": prediction_containers['test']
    }

In [ ]:
def random_search(param_distributions, n_iter):
    """Generate random hyperparameter combinations for grid search."""
    param_list = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=42))
    return param_list

def run_random_search(raw_filename, n_iter=10):
    """
    Perform random hyperparameter search for GCN model optimization.
    
    Args:
        raw_filename (str): Path to the dataset CSV file
        n_iter (int): Number of parameter combinations to test
        
    Returns:
        tuple: Best parameters and corresponding MSE score
    """
    # Define the hyperparameter search space
    param_distributions = {
        'radius': [2, 3, 4, 5, 6],
        'fingerprint_dim': [30, 70, 150, 210, 300],
        'p_dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
        'weight_decay': [2, 3, 4, 5, 6],
        'learning_rate': [2, 3, 4, 5],
    }

    # Fixed parameters for training
    fixed_params = {
        'T': 2,  # Number of iterations for message passing
        'batch_size': 50,
        'epochs': 800
    }

    # Generate random hyperparameter combinations
    param_list = random_search(param_distributions, n_iter)

    best_mse = float('inf')
    best_params = None

    for params in param_list:
        print(f"Testing parameters: {params}")

        # Combine fixed and variable parameters
        current_params = {**fixed_params, **params}

        # Prepare model and data with current hyperparameters
        model, optimizer, loss_function, remained_df, targets, feature_dicts = prepare_model_and_data(
            raw_filename, **current_params
        )

        # Train and evaluate model with current parameters
        result = train_and_evaluate(
            model, remained_df, targets, feature_dicts, optimizer, loss_function,
            fixed_params['batch_size'], fixed_params['epochs']
        )

        # Extract test MSE from results
        test_mse = result["overall_metrics"][3]
        print(f"Test MSE: {test_mse:.4f}")

        if test_mse < best_mse:
            best_mse = test_mse
            best_params = params

    print(f"Best parameters: {best_params}")
    print(f"Best MSE: {best_mse}")

    return best_params, best_mse

if __name__ == "__main__":
    raw_filename = "/notebooks/Codebase/Database/cal_abs.csv"
    best_params, best_mse = run_random_search(raw_filename, n_iter=10)

Testing parameters: {'weight_decay': 2, 'radius': 4, 'p_dropout': 0.5, 'learning_rate': 4, 'fingerprint_dim': 70}
number of all smiles:  38
number of successfully processed smiles:  38
Early stopping at epoch 76

Fold 1 Summary:
Train MSE: 3.0566
Val MSE: 7.4403
Test MSE: 11.2821
Early stopping at epoch 31

Fold 2 Summary:
Train MSE: 3.3897
Val MSE: 7.6116
Test MSE: 6.6658
Early stopping at epoch 31

Fold 3 Summary:
Train MSE: 4.4283
Val MSE: 3.2075
Test MSE: 4.1222
Early stopping at epoch 42

Fold 4 Summary:
Train MSE: 4.4154
Val MSE: 3.1804
Test MSE: 3.8248
Early stopping at epoch 42

Fold 5 Summary:
Train MSE: 4.1395
Val MSE: 4.4577
Test MSE: 4.3671
Early stopping at epoch 49

Fold 6 Summary:
Train MSE: 4.2670
Val MSE: 4.5470
Test MSE: 1.8580
Early stopping at epoch 71

Fold 7 Summary:
Train MSE: 3.4190
Val MSE: 5.0870
Test MSE: 25.9288
Early stopping at epoch 32

Fold 8 Summary:
Train MSE: 4.1172
Val MSE: 3.9783
Test MSE: 10.5875
Early stopping at epoch 34

Fold 9 Summary:
Train MS